In [1]:
import pandas as pd
import sys
import numpy as np
import time
from IPython.display import display
import dataframe_image as dfi


%load_ext autoreload
%autoreload 2

# 1. Combined Import (Import everything once)
from TES_Availability_Engine import (
    calculate_thermal_potentials as calc_combined, 
    PV_SETTINGS, 
    HP_HEATING_SETTINGS, 
    HP_COOLING_SETTINGS
)

MAX_COP_LIMIT_H = HP_HEATING_SETTINGS['MAX_COP']
MAX_COP_LIMIT_C = HP_COOLING_SETTINGS['MAX_COP']


# =================================================================
# 1. USER INPUT SECTION (COMBINED STRATEGIES)
# =================================================================
thermal_data_file = 'Input - Results_Yearly_Zone_Demands_Combined no vent.csv' 
materials_file    = 'Input - Multi zone simplified thermal mass - case study.txt'

# Shared Building Settings
T_START_LIST = [21.0, 22.0, 21.0, 21.0, 21.0]                       # Initial temperature for each zone (°C).
T_SET_LIST   = [21.0, 22.0, 21.0, 21.0, 21.0]                       # Setpoint temperature for each zone (°C).
T_SWING_LIST = [2.0, 0.0, 2.0, 2.0, 2.0]                          # Allowed temperature swing for each zone (°C) - defines the comfort band around the setpoint.

# --- TWO SEPARATE HEAT PUMP CAPACITIES ---
HP_MAX_CAPACITY_H = 115.0                                # Max kW for the Heating HP.
HP_MAX_CAPACITY_C = 65.0                                # Max kW for the Cooling HP.

# --- HOT TANK (Heating) Settings ---
REHEAT_STRATEGY_H    = "proportional"                   # For Thermal Mass Charging (Surplus) - sequential or proportional.
REHEAT_PRIORITY_H    = [1, 2, 3, 4, 5]                        # Priority order specifically for Charging.
DISCHARGE_STRATEGY_H = "proportional"                   # For Modulated HP Distribution (Congestion) - sequential or proportional.
DISCHARGE_PRIORITY_H = [1, 2, 3, 4, 5]                        # Priority order specifically for Discharging  .     
ZONE_TES_FIRST_H     = [False, False, False, False, False]             # True = Use TES to keep T_SET immediately. False = Drift to T_MIN first.
delta_t_H            = 10.0                             # Temperature difference for Heating.
MANUAL_TANK_KWH_H = None                                # Set a specific tank size in kWh to bypass optimization (e.g., 500.0), or None to run the optimization loop.

# --- COLD TANK (Cooling) Settings ---
REHEAT_STRATEGY_C    = "proportional"                   # For Thermal Mass Charging (Surplus) - sequential or proportional.
REHEAT_PRIORITY_C    = [1, 2, 3, 4, 5]                        # Priority order specifically for Charging.
DISCHARGE_STRATEGY_C = "proportional"                   # For Modulated HP Distribution (Congestion) - sequential or proportional.
DISCHARGE_PRIORITY_C = [1, 2, 3, 4, 5]                        # Priority order specifically for Discharging.  
ZONE_TES_FIRST_C     = [False, False, False, False, False]             # True = Use TES to keep T_SET immediately. False = Drift to T_MAX first.
delta_t_C            = 8.0                              # Temperature difference for Cooling.
MANUAL_TANK_KWH_C = None                                # Set a specific tank size in kWh to bypass optimization (e.g., 500.0), or None to run the optimization loop.

# --- HYDRAULIC INPUT ---
ZONES_DELTA_T_H = [10.0, 10.0, 10.0, 10.0, 10.0]                    # Temperature difference per zone (Delta T between supply and return).
ZONES_DELTA_T_C = [8.0, 8.0, 8.0, 8.0, 8.0]                       # Temperature difference per zone (Delta T between supply and return).

# --- SHARED CONSTANTS ---
S, rho, cp, eta_str = 1.2, 1000, 4.18, 0.95             # Constants: Safety factor, density of water (kg/m3), specific heat capacity of water (kWh/kg.K), and stratification efficiency of the tank.
DEAD_ZONE = 0.20                                        # Unusable fraction of the tank capacity.
TAU_STABILITY = 0.25                                    # Minimum fraction of the timestep that must be stable to avoid oscillation penalties.
TOTAL_EFFICIENCY = eta_str * (1 - DEAD_ZONE)
UTILIZATION_FACTOR = 1.2

# --- HP HARDWARE & GRID HANDSHAKE ---
KVA_SPLIT_MODE = 'proportional'                         # Options: 'fixed' or 'proportional'.
HEATING_SHARE_INPUT = 0.5                               # Master Knob: % of grid/PV to this tank when selected 'fixed' mode for KVA_SPLIT_MODE.
HP_MIN_MOD_KVA = 1.5                                    # 5% Electrical Maintenance Floor.
DRIFT_RATE = 0.0001                                     # Hourly thermal loss (e.g., 0.01%).

MIN_MODULATION_KVA_H = HP_MIN_MOD_KVA                   # Minimum modulation threshold for the Heating HP (kVA).
MIN_MODULATION_KVA_C = HP_MIN_MOD_KVA                   # Minimum modulation threshold for the Cooling HP (kVA).

# --- Extra display options ---
SHOW_FLOW_RATES = True                                  # Set to 'True' to show the flowrates per zone. Set to 'False' to hide them.

# =================================================================
# 2. DATA LOADING & PRE-PROCESSING (COMBINED)
# =================================================================
try:
    # 1. Thermal Mass Calculation (Same as before)
    df_mats = pd.read_csv(materials_file) 
    df_mats['kWh_per_K'] = (df_mats['FloorArea'] * df_mats['SpecCapacity'] * 1000) / 3.6 / 1_000_000
    zone_masses = df_mats['kWh_per_K'].tolist()
    
    # 2. Load the Combined Demand File
    df_input = pd.read_csv(thermal_data_file)

    # --- CRITICAL: Handle NaNs in demands to prevent Bisection Violations ---                              
    # We fill with 0 to ensure the optimization engine doesn't break on empty cells
    df_input = df_input.fillna(0)                                                                                      # ADDED

    df_input['Hour'] = df_input.index + 1 
    demand_cols = [col for col in df_input.columns if 'Demand' in col]

    #Dynamically determine amount of zones and adjust simulation for n zones based on the columns in the input file. This allows for flexibility in the number of zones without hardcoding.
    num_zones = len(demand_cols)

    # Ensure User Inputs match the file dimensions (Safety Check)
    if len(T_START_LIST) != num_zones:
        print(f"Warning: T_START_LIST has {len(T_START_LIST)} items but file has {num_zones} zones. Adjusting...")

    # 3. Load Weather and Grid (Same as before)
    df_solar = pd.read_csv('Input PV - NEN5060-B2 1%.txt')
    df_temp  = pd.read_csv('Input Temp - NEN5060-B2 1%.txt')
    df_grid  = pd.read_csv('Output - total kVA - Hour, Grid_kVA.csv')

    # Ensure lengths match before copying to avoid index shifting
    sim_length = len(df_input)                                                                                          # ADDED

    # 1. Align Grid (Loops your 168-day file to fill the full year)
    grid_indices = np.arange(sim_length) % len(df_grid)
    df_grid_aligned = df_grid.iloc[grid_indices].reset_index(drop=True)

    # 2. Align Weather (Solar)
    solar_indices = np.arange(sim_length) % len(df_solar)
    df_weather = df_solar.iloc[solar_indices].reset_index(drop=True)

    # 3. Align Weather (Temperature)
    temp_indices = np.arange(sim_length) % len(df_temp)
    df_weather['T'] = df_temp['T'].values[temp_indices]

    # 4. Identify Heating and Cooling Demands for the Engine Split per timestep
    # This is the total heating and total cooling demand per timestep, which will be used by the engine to determine how to split the available kVA.
    h_dem_series = df_input[demand_cols].apply(lambda x: x[x > 0].sum(), axis=1)
    c_dem_series = df_input[demand_cols].apply(lambda x: abs(x[x < 0].sum()), axis=1)

    # --- START ENGINE INTEGRATION (The Combined Handshake) ---
    df_potentials = calc_combined(
    df_weather, 
    df_grid_aligned, 
    split_mode=KVA_SPLIT_MODE, 
    h_demand=h_dem_series, 
    c_demand=c_dem_series,
    manual_share=HEATING_SHARE_INPUT,
    hp_max_h=HP_MAX_CAPACITY_H,
    hp_max_c=HP_MAX_CAPACITY_C
)

    # 5. Map both Heating AND Cooling potentials to the input dataframe
    # We maintain separate columns for each side's constraints
    df_input['Available_kVA_H'] = df_potentials['Elec_kVA_Heating'].values
    df_input['COP_H']           = df_potentials['COP_Heating'].values
    df_input['Max_Supply_kW_H'] = df_potentials['Potential_Heating_kW'].clip(upper=HP_MAX_CAPACITY_H).values

    df_input['Available_kVA_C'] = df_potentials['Elec_kVA_Cooling'].values
    df_input['COP_C']           = df_potentials['COP_Cooling'].values
    df_input['Max_Supply_kW_C'] = df_potentials['Potential_Cooling_kW'].clip(upper=HP_MAX_CAPACITY_C).values
    # --- END ENGINE INTEGRATION ---

except Exception as e:
    sys.exit(f"File Error: {e}")

# =================================================================
# 3. SIMULATION ENGINE (PARALLEL DUAL-HP MODE)
# =================================================================

def simulate_merged_multi_zone(
    df_input, demand_cols, zone_masses, T_starts, T_sets, T_swings, 
    tank_cap_h, tank_cap_c,
    # --- Heating ---
    reh_strat_h, dis_strat_h, reh_prio_h, dis_prio_h, tes_h, zone_dt_list_h,
    # --- Cooling ---
    reh_strat_c, dis_strat_c, reh_prio_c, dis_prio_c, tes_c, zone_dt_list_c,
    # --- Constants ---
    min_mod_kva_h, min_mod_kva_c, drift_rate
):
    
    # Initialization and Boundaries
    T_mins = [s - sw for s, sw in zip(T_sets, T_swings)]
    T_maxs = [s + sw for s, sw in zip(T_sets, T_swings)]
    current_temps = list(T_starts)
    current_tank_kwh_h = tank_cap_h if MANUAL_TANK_KWH_H is None else MANUAL_TANK_KWH_H
    current_tank_kwh_c = tank_cap_c if MANUAL_TANK_KWH_C is None else MANUAL_TANK_KWH_C

    EPSILON = 1e-7 
    num_steps = len(df_input)
    num_zones = len(demand_cols)

    # Transform pandas to NumPy arrays for faster access in the loop
    hour_arr = df_input['Hour'].values
    demand_arr = df_input[demand_cols].values

    # Heating Infrastructure Data
    max_supply_arr_h = df_input['Max_Supply_kW_H'].values
    cop_arr_h = df_input['COP_H'].values
    available_kva_arr_h = df_input['Available_kVA_H'].values
    cong_indices_h = np.where(max_supply_arr_h < (np.where(demand_arr > 0, demand_arr, 0).sum(axis=1) - EPSILON))[0]
    next_cong_map_h = np.searchsorted(cong_indices_h, np.arange(num_steps), side='right')

    # Cooling Infrastructure Data
    max_supply_arr_c = df_input['Max_Supply_kW_C'].values
    cop_arr_c = df_input['COP_C'].values
    available_kva_arr_c = df_input['Available_kVA_C'].values
    cong_indices_c = np.where(max_supply_arr_c < (np.where(demand_arr < 0, abs(demand_arr), 0).sum(axis=1) - EPSILON))[0]
    next_cong_map_c = np.searchsorted(cong_indices_c, np.arange(num_steps), side='right')

    # Prep Matrix (Adjusted for dual results)
    num_base_cols = 9
    num_per_zone = 4 if SHOW_FLOW_RATES else 3
    num_individual_cols = num_base_cols + (num_zones * num_per_zone)

    # We create TWO separate matrices so they match your individual script output
    results_matrix_H = np.zeros((num_steps, num_individual_cols))
    results_matrix_C = np.zeros((num_steps, num_individual_cols))

    strat_h_hist, strat_c_hist = [], []

    # Define headers (to be used at the very end of the function)
    headers_individual = ["Sim_Timestep", "Total_Demand_kW", "Max_Supply_kW", "HP_Out_kW", "Tank_kWh", "Deficit_kW", "Window_h", "Tsat_TES"]
    for i in range(1, num_zones + 1):
        headers_individual += [f"Demand_Z{i}_kW", f"Temp_Z{i}", f"Tsat_Z{i}_h"]
        if SHOW_FLOW_RATES:
            headers_individual.append(f"Flow_Z{i}_m3h")

    # -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    for idx in range(num_steps):
        # A. Drift (Both tanks lose potential)
        current_tank_kwh_h *= (1 - drift_rate)
        current_tank_kwh_c *= (1 - drift_rate)

        demands = demand_arr[idx]
        current_time_val = hour_arr[idx]
        
        # --- Split demands for logic processing ---
        # Separates the combined demand array into positive (heating) and absolute negative (cooling) values            # Hier gebleven door dit toe te voegen. Wat nog meer?
        h_demands = np.maximum(0, demands)
        c_demands = np.abs(np.minimum(0, demands))

        # Per timestep, the script knows how much total heating and cooling demand there is.
        total_demand_h = np.sum(h_demands)
        total_demand_c = np.sum(c_demands)

        # --- MATCHING YOUR INDIVIDUAL CODE INITIALIZATION ---
        # 1. Constants for the Floor calculation (Split for H and C)
        available_kva_limit_h = available_kva_arr_h[idx]
        available_kva_limit_c = available_kva_arr_c[idx]
        current_cop_h = cop_arr_h[idx]
        current_cop_c = cop_arr_c[idx]
        
        # 2. Strategy Variables (Initialized to 0.0)
        delta_t_window_h, tsat_tes_h, tsat_struct_h = 0.0, 0.0, 0.0
        delta_t_window_c, tsat_tes_c, tsat_struct_c = 0.0, 0.0, 0.0
        
        tsats_per_zone_h = [0.0] * num_zones
        tsats_per_zone_c = [0.0] * num_zones
        
        active_strat_h = "Discharge"
        active_strat_c = "Discharge"
        
        current_zone_flows_h = [0.0] * num_zones
        current_zone_flows_c = [0.0] * num_zones

        # 3. Energy Transfer Loggers
        e_to_zones_h, e_to_tank_h = 0.0, 0.0
        e_to_zones_c, e_to_tank_c = 0.0, 0.0

        # 4. INITIALIZE FLOW LISTS FOR THIS TIMESTEP
        zone_flows_h = []
        zone_flows_c = []

        actual_hp_out_h = 0.0
        actual_hp_out_c = 0.0

        # --- Violation Tracking (Required for Red Tables) ---
        deficit_to_track_h = 0.0
        deficit_to_track_c = 0.0

        results_matrix_H[idx, 0] = current_time_val
        results_matrix_H[idx, 1] = total_demand_h
        results_matrix_C[idx, 1] = total_demand_c

        # =================================================================
        # B. HEATING HP BRANCH (MATCHES HOT TANK V7 STRUCTURE)
        # =================================================================
        available_power_h = max_supply_arr_h[idx]
        
        if available_power_h >= (total_demand_h - EPSILON):
            # REHEAT / SURPLUS MODE
            current_zone_flows_h = [(d * 3600) / (rho * cp * zone_dt_list_h[i]) if d > EPSILON else 0.0 for i, d in enumerate(h_demands)]
            surplus_h = max(0.0, available_power_h - total_demand_h)
            actual_hp_out_h = total_demand_h + surplus_h
            
            map_idx_h = next_cong_map_h[idx]
            if map_idx_h < len(cong_indices_h):
                next_event_idx_h = cong_indices_h[map_idx_h]
                delta_t_window_h = float(hour_arr[next_event_idx_h] - current_time_val)
            else:
                delta_t_window_h = float(hour_arr[-1] - current_time_val)
            
            # AGNOSTIC HUNGER: Energy gap to reach T_set (Works for both Heat/Cool)
            hunger_h = [max(0, T_sets[i] - current_temps[i]) * zone_masses[i] for i in range(num_zones)]
            total_hunger_h = sum(hunger_h)

            tsat_struct_h = (total_hunger_h / surplus_h * UTILIZATION_FACTOR) if surplus_h > EPSILON else 0.0                                # ADDED EPISLON instead of 0
            tsats_per_zone_h = [(h / surplus_h * UTILIZATION_FACTOR) if surplus_h > EPSILON else 0.0 for h in hunger_h]
            
            tank_gap_h = max(0, tank_cap_h - current_tank_kwh_h)
            tsat_tes_h = (tank_gap_h / surplus_h) if surplus_h > EPSILON else 0.0 # ADDED

            # HIERARCHICAL STRATEGY
            if delta_t_window_h >= (tsat_struct_h - EPSILON):
                active_strat_h = "STRUCT"
                e_to_zones_h = min(surplus_h, total_hunger_h)
                e_to_tank_h = min(surplus_h - e_to_zones_h, tank_gap_h)
            else:
                active_strat_h = "TES"
                e_to_tank_h = min(surplus_h, tank_gap_h)
                e_to_zones_h = min(surplus_h - e_to_tank_h, total_hunger_h)
            
            # AGNOSTIC TEMPERATURE UPDATE
            if e_to_zones_h > EPSILON:
                if reh_strat_h == "proportional":
                    for i in range(num_zones):
                        if hunger_h[i] > EPSILON:
                            share_surplus_h = hunger_h[i] / total_hunger_h
                            zone_pwr_share = share_surplus_h * e_to_zones_h
                            tsats_per_zone_h[i] = (hunger_h[i] / zone_pwr_share * UTILIZATION_FACTOR)
                            current_temps[i] += (zone_pwr_share / zone_masses[i])
                            current_zone_flows_h[i] = ((h_demands[i] + zone_pwr_share) * 3600) / (rho * cp * zone_dt_list_h[i])
                
                elif reh_strat_h == "sequential":
                    remaining_e_h = e_to_zones_h
                    for p_idx in reh_prio_h:
                        i = p_idx - 1
                        if hunger_h[i] > EPSILON and remaining_e_h > EPSILON:
                            give_h = min(remaining_e_h, hunger_h[i])
                            tsats_per_zone_h[i] = (hunger_h[i] / give_h * UTILIZATION_FACTOR)
                            current_temps[i] += (give_h / zone_masses[i])
                            current_zone_flows_h[i] = ((h_demands[i] + give_h) * 3600) / (rho * cp * zone_dt_list_h[i])
                            remaining_e_h -= give_h
            
            current_tank_kwh_h += e_to_tank_h
            
        else:
            # DISCHARGE MODE
            actual_hp_out_h = max(min_mod_kva_h * current_cop_h, available_power_h) if total_demand_h > 0 else 0.0
            hp_shares_kw_h = [0.0] * num_zones
            
            if total_demand_h > EPSILON:
                if dis_strat_h == "proportional":
                    for i in range(num_zones):
                        # Explicitly calculate the share of total heating demand
                        share_h = h_demands[i] / total_demand_h
                        hp_shares_kw_h[i] = actual_hp_out_h * share_h
                        
                elif dis_strat_h == "sequential":
                    rem_hp = actual_hp_out_h
                    for p_idx in dis_prio_h:
                        i = p_idx - 1
                        give_h = min(rem_hp, h_demands[i])
                        hp_shares_kw_h[i] = give_h
                        rem_hp -= give_h
            
            for i in range(num_zones):
                if h_demands[i] > EPSILON:
                    deficit = h_demands[i] - hp_shares_kw_h[i]
                    actual_from_tank = 0.0
                    
                    if tes_h[i]:
                        # --- MODIFIED: Only discharge tank if we are violating or near violation ---
                        # We allow drift until we hit T_mins[i]
                        if current_temps[i] <= (T_mins[i] + EPSILON):
                            actual_from_tank = min(deficit, current_tank_kwh_h)
                            current_tank_kwh_h -= actual_from_tank
                        
                        unmet_energy_h = deficit - actual_from_tank
                        if unmet_energy_h > EPSILON:
                            deficit_to_track_h += unmet_energy_h
                            
                        # Physical drift: Temp drops because demand > supply
                        current_temps[i] -= unmet_energy_h / zone_masses[i]
                    else:
                        # Existing logic for non-TES zones...
                        t_pred = current_temps[i] - (h_demands[i] / zone_masses[i]) + (hp_shares_kw_h[i] / zone_masses[i])
                        if t_pred < (T_mins[i] - EPSILON):
                            needed = (T_mins[i] - t_pred) * zone_masses[i]
                            actual_from_tank = min(needed, current_tank_kwh_h)
                            current_tank_kwh_h -= actual_from_tank
                            current_temps[i] = t_pred + (actual_from_tank / zone_masses[i])
                            if current_temps[i] < (T_mins[i] - EPSILON):
                                deficit_to_track_h += (T_mins[i] - current_temps[i]) * zone_masses[i]
                        else:
                            current_temps[i] = t_pred
                    
                    zone_pwr = hp_shares_kw_h[i] + actual_from_tank
                    current_zone_flows_h[i] = (zone_pwr * 3600) / (rho * cp * zone_dt_list_h[i])     

        # =================================================================
        # C. COOLING HP BRANCH (EXACT MIRROR OF HEATING LOGIC)
        # =================================================================
        available_power_c = max_supply_arr_c[idx]
        
        if available_power_c >= (total_demand_c - EPSILON):
            # --- REHEAT / SURPLUS MODE (COOLING) ---
            # Initial flow based on hourly demand
            current_zone_flows_c = [(d * 3600) / (rho * cp * zone_dt_list_c[i]) if d > EPSILON else 0.0 for i, d in enumerate(c_demands)]               # ADDED EPISLON instead of 0
            surplus_c = max(0.0, available_power_c - total_demand_c)
            actual_hp_out_c = total_demand_c + surplus_c
            
            # Identify next congestion for delta_t_window
            map_idx_c = next_cong_map_c[idx]
            if map_idx_c < len(cong_indices_c):
                delta_t_window_c = float(hour_arr[cong_indices_c[map_idx_c]] - current_time_val)
            else:
                delta_t_window_c = float(hour_arr[-1] - current_time_val)
            
            # AGNOSTIC HUNGER: Energy gap to reach T_set from above
            hunger_c = [max(0, current_temps[i] - T_sets[i]) * zone_masses[i] for i in range(num_zones)]
            total_hunger_c = sum(hunger_c)

            tsat_struct_c = (total_hunger_c / surplus_c * UTILIZATION_FACTOR) if surplus_c > EPSILON else 0.0
            tsats_per_zone_c = [(h / surplus_c * UTILIZATION_FACTOR) if surplus_c > EPSILON else 0.0 for h in hunger_c]

            tank_gap_c = max(0, tank_cap_c - current_tank_kwh_c)
            tsat_tes_c = (tank_gap_c / surplus_c) if surplus_c > EPSILON else 0.0 # ADDED

            # HIERARCHICAL STRATEGY (Exact same logic as Heating)
            if delta_t_window_c >= (tsat_struct_c - EPSILON):
                active_strat_c = "STRUCT"
                e_to_zones_c = min(surplus_c, total_hunger_c)
                e_to_tank_c = min(surplus_c - e_to_zones_c, tank_gap_c)
            else:
                active_strat_c = "TES"
                e_to_tank_c = min(surplus_c, tank_gap_c)
                e_to_zones_c = min(surplus_c - e_to_tank_c, total_hunger_c)
            
            # COOLING TEMPERATURE & FLOW UPDATE
            if e_to_zones_c > EPSILON:
                if reh_strat_c == "proportional":
                    for i in range(num_zones):
                        if hunger_c[i] > EPSILON:
                            share_surplus_c = hunger_c[i] / total_hunger_c
                            zone_pwr_share_c = share_surplus_c * e_to_zones_c
                            current_temps[i] -= (zone_pwr_share_c / zone_masses[i])
                            current_zone_flows_c[i] = ((c_demands[i] + zone_pwr_share_c) * 3600) / (rho * cp * zone_dt_list_c[i])
                
                elif reh_strat_c == "sequential":
                    remaining_e_c = e_to_zones_c
                    for p_idx in reh_prio_c:
                        i = p_idx - 1
                        if hunger_c[i] > EPSILON and remaining_e_c > EPSILON:
                            give_c = min(remaining_e_c, hunger_c[i])
                            current_temps[i] -= (give_c / zone_masses[i])
                            current_zone_flows_c[i] = ((c_demands[i] + give_c) * 3600) / (rho * cp * zone_dt_list_c[i])
                            remaining_e_c -= give_c
            
            current_tank_kwh_c += e_to_tank_c
            
        else:
            # --- DISCHARGE MODE (COOLING) ---
            actual_hp_out_c = max(min_mod_kva_c * current_cop_c, available_power_c) if total_demand_c > 0 else 0.0
            hp_shares_kw_c = [0.0] * num_zones
            
            if total_demand_c > EPSILON:
                if dis_strat_c == "proportional":
                    for i in range(num_zones):
                        share_c = c_demands[i] / total_demand_c
                        hp_shares_kw_c[i] = actual_hp_out_c * share_c
                
                elif dis_strat_c == "sequential":
                    rem_hp_c = actual_hp_out_c
                    for p_idx in dis_prio_c:
                        i = p_idx - 1
                        give_c = min(rem_hp_c, c_demands[i])
                        hp_shares_kw_c[i] = give_c
                        rem_hp_c -= give_c
            
            for i in range(num_zones):
                if c_demands[i] > EPSILON:
                    deficit = c_demands[i] - hp_shares_kw_c[i]
                    actual_from_tank = 0.0
                    
                    if tes_c[i]:
                        actual_from_tank = min(deficit, current_tank_kwh_c)
                        current_tank_kwh_c -= actual_from_tank
                        # Track what the tank couldn't cover
                        unmet_energy_c = deficit - actual_from_tank
                        if unmet_energy_c > EPSILON:
                            deficit_to_track_c += unmet_energy_c
                        # Temp rises because demand > supply
                        current_temps[i] += unmet_energy_c / zone_masses[i]
                    else:
                        t_pred = current_temps[i] + (c_demands[i] / zone_masses[i]) - (hp_shares_kw_c[i] / zone_masses[i])
                        if t_pred > (T_maxs[i] + EPSILON):
                            needed = (t_pred - T_maxs[i]) * zone_masses[i]
                            actual_from_tank = min(needed, current_tank_kwh_c)
                            current_tank_kwh_c -= actual_from_tank
                            current_temps[i] = t_pred - (actual_from_tank / zone_masses[i])
                            
                            # --- (Tank was used but not enough) ---
                            if current_temps[i] > (T_maxs[i] + EPSILON):
                                deficit_to_track_c += (current_temps[i] - T_maxs[i]) * zone_masses[i]
                        else:
                            current_temps[i] = t_pred
                    
                    # Re-assign the result to the existing cooling array
                    zone_pwr = hp_shares_kw_c[i] + actual_from_tank
                    current_zone_flows_c[i] = (zone_pwr * 3600) / (rho * cp * zone_dt_list_c[i])

        # -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

        # --- HEATING LOGGING (Populate Results Matrix H) ---
        results_matrix_H[idx, 0:9] = [
            current_time_val, total_demand_h, total_demand_c, available_power_h, actual_hp_out_h, 
            current_tank_kwh_h, deficit_to_track_h, delta_t_window_h, tsat_tes_h
        ]
        strat_h_hist.append(active_strat_h)
        
        col_ptr_h = 9
        for i in range(num_zones):
            results_matrix_H[idx, col_ptr_h]     = h_demands[i]
            results_matrix_H[idx, col_ptr_h + 1] = current_temps[i]
            results_matrix_H[idx, col_ptr_h + 2] = tsats_per_zone_h[i]
            col_ptr_h += 3
            if SHOW_FLOW_RATES:
                results_matrix_H[idx, col_ptr_h] = current_zone_flows_h[i]
                col_ptr_h += 1

        # --- COOLING LOGGING (Populate Results Matrix C) ---
        results_matrix_C[idx, 0:9] = [
            current_time_val, total_demand_h, total_demand_c, available_power_c, actual_hp_out_c, 
            current_tank_kwh_c, deficit_to_track_c, delta_t_window_c, tsat_tes_c
        ]
        strat_c_hist.append(active_strat_c)
        
        col_ptr_c = 9
        for i in range(num_zones):
            results_matrix_C[idx, col_ptr_c]     = c_demands[i]
            results_matrix_C[idx, col_ptr_c + 1] = current_temps[i]
            results_matrix_C[idx, col_ptr_c + 2] = tsats_per_zone_c[i]
            col_ptr_c += 3
            if SHOW_FLOW_RATES:
                results_matrix_C[idx, col_ptr_c] = current_zone_flows_c[i]
                col_ptr_c += 1

    # -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    # Define Column Headers
    cols = ["Sim_Timestep", "Total_Demand_H_kW", "Total_Demand_C_kW", "Max_Supply_kW", "HP_Out_kW", "Tank_kWh", "Deficit_kW", "Window_h", "Tsat_TES"]
    for i in range(num_zones):
        cols.extend([f"Demand_Z{i+1}_kW", f"Temp_Z{i+1}", f"Tsat_Z{i+1}_h"])
        if SHOW_FLOW_RATES:
            cols.append(f"Flow_Z{i+1}_m3h")

    # Create Heating DataFrame
    df_h = pd.DataFrame(results_matrix_H, columns=cols)
    df_h["Strategy"] = strat_h_hist
    
    # Create Cooling DataFrame (using same col names for compatibility)
    df_c = pd.DataFrame(results_matrix_C, columns=cols)
    df_c["Strategy"] = strat_c_hist

    return df_h, df_c, T_mins, T_maxs

# =================================================================
# 4. RUN & OPTIMIZE (BISECTION SEARCH) & VIOLATIONS CHECK
# =================================================================

def check_violations(df, T_mins, T_maxs, T_sets, tes_first_list):
    """
    Checks for thermal violations across both heating and cooling demands.
    - For Heating: Temp must not drop below (T_set if TES-first, else T_min).
    - For Cooling: Temp must not rise above (T_set if TES-first, else T_max).
    """
    for i, (t_min, t_max, t_set, tes_first) in enumerate(zip(T_mins, T_maxs, T_sets, tes_first_list)):
        # --- Heating Violation Check ---
        # If TES-first, floor is the setpoint; otherwise, it's the absolute minimum.
        floor = t_set if tes_first else t_min
        if (df[f"Temp_Z{i+1}"] < (floor - 1e-6)).any():
            return True
            
        # --- Cooling Violation Check ---
        # If TES-first, ceiling is the setpoint; otherwise, it's the absolute maximum.
        ceiling = t_set if tes_first else t_max
        if (df[f"Temp_Z{i+1}"] > (ceiling + 1e-6)).any():
            return True
            
    return False

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

T_mins_calc = [s - sw for s, sw in zip(T_SET_LIST, T_SWING_LIST)]
T_maxs_calc = [s + sw for s, sw in zip(T_SET_LIST, T_SWING_LIST)]

start_total = time.perf_counter()
run_count = 0

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- 4.1 HOT TANK OPTIMIZATION (HEATING) ---
if MANUAL_TANK_KWH_H is None:
    print("Optimizing Hot Tank Size (Bisection Method)...")
    low_h, high_h = 0.0, 5000.0  # Search range in kWh
    best_tank_h = high_h
    
    for i in range(15):
        mid_h = (low_h + high_h) / 2
        # Run sim: Testing mid_h for heating, keeping cooling tank at 0 (or a safe high value)
        df_h, _, t_mins, t_maxs = simulate_merged_multi_zone(
            df_input, demand_cols, zone_masses, T_START_LIST, T_SET_LIST, T_SWING_LIST,
            mid_h, 5000.0, # Testing heating specifically
            REHEAT_STRATEGY_H, DISCHARGE_STRATEGY_H, REHEAT_PRIORITY_H, DISCHARGE_PRIORITY_H, ZONE_TES_FIRST_H, ZONES_DELTA_T_H,
            REHEAT_STRATEGY_C, DISCHARGE_STRATEGY_C, REHEAT_PRIORITY_C, DISCHARGE_PRIORITY_C, ZONE_TES_FIRST_C, ZONES_DELTA_T_C,
            MIN_MODULATION_KVA_H, MIN_MODULATION_KVA_C, DRIFT_RATE
        )
        
        # Check violations in Heating DataFrame
        if check_violations(df_h, t_mins, t_maxs, T_SET_LIST, ZONE_TES_FIRST_H):
            low_h = mid_h
        else:
            high_h = mid_h
            best_tank_h = mid_h
        print(f"  Iter {i+1}: Testing {mid_h:.2f} kWh -> {'Violations' if low_h == mid_h else 'Pass'}")
    
    TANK_KWH_H_FINAL = best_tank_h
else:
    TANK_KWH_H_FINAL = MANUAL_TANK_KWH_H
    print(f"Using Manual Hot Tank Size: {TANK_KWH_H_FINAL} kWh")

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- 4.2 COLD TANK OPTIMIZATION (COOLING) ---
if MANUAL_TANK_KWH_C is None:
    print("\nOptimizing Cold Tank Size (Bisection Method)...")
    low_c, high_c = 0.0, 5000.0
    best_tank_c = high_c
    
    for i in range(15):
        mid_c = (low_c + high_c) / 2
        # Run sim: Using the FINAL hot tank size, testing mid_c for cooling
        _, df_c, t_mins, t_maxs = simulate_merged_multi_zone(
            df_input, demand_cols, zone_masses, T_START_LIST, T_SET_LIST, T_SWING_LIST,
            TANK_KWH_H_FINAL, mid_c, 
            REHEAT_STRATEGY_H, DISCHARGE_STRATEGY_H, REHEAT_PRIORITY_H, DISCHARGE_PRIORITY_H, ZONE_TES_FIRST_H, ZONES_DELTA_T_H,
            REHEAT_STRATEGY_C, DISCHARGE_STRATEGY_C, REHEAT_PRIORITY_C, DISCHARGE_PRIORITY_C, ZONE_TES_FIRST_C, ZONES_DELTA_T_C,
            MIN_MODULATION_KVA_H, MIN_MODULATION_KVA_C, DRIFT_RATE
        )
        
        # Check violations in Cooling DataFrame
        if check_violations(df_c, t_mins, t_maxs, T_SET_LIST, ZONE_TES_FIRST_C):
            low_c = mid_c
        else:
            high_c = mid_c
            best_tank_c = mid_c
        print(f"  Iter {i+1}: Testing {mid_c:.2f} kWh -> {'Violations' if low_c == mid_c else 'Pass'}")
    
    TANK_KWH_C_FINAL = best_tank_c
else:
    TANK_KWH_C_FINAL = MANUAL_TANK_KWH_C
    print(f"Using Manual Cold Tank Size: {TANK_KWH_C_FINAL} kWh")

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- 4.3 FINAL RUN & PERFORMANCE SUMMARY ---
run_count += 1
df_res_h, df_res_c, final_t_mins, final_t_maxs = simulate_merged_multi_zone(
    df_input, demand_cols, zone_masses, T_START_LIST, T_SET_LIST, T_SWING_LIST,
    TANK_KWH_H_FINAL, TANK_KWH_C_FINAL,
    REHEAT_STRATEGY_H, DISCHARGE_STRATEGY_H, REHEAT_PRIORITY_H, DISCHARGE_PRIORITY_H, ZONE_TES_FIRST_H, ZONES_DELTA_T_H,
    REHEAT_STRATEGY_C, DISCHARGE_STRATEGY_C, REHEAT_PRIORITY_C, DISCHARGE_PRIORITY_C, ZONE_TES_FIRST_C, ZONES_DELTA_T_C,
    MIN_MODULATION_KVA_H, MIN_MODULATION_KVA_C, DRIFT_RATE
)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

end_total = time.perf_counter()
print(f"\nOptimization Complete.")
print(f"Total Simulation Runs: {run_count}")
print(f"Total Time Taken: {end_total - start_total:.2f} seconds")
print("-" * 33)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- HYDRAULIC STABILITY CALCULATION (Heating Side - Vh_h) ---
df_res_h['Net_Power_Deficit_kW'] = (df_res_h['Total_Demand_H_kW'] - df_res_h['HP_Out_kW']).clip(lower=0)
df_res_h['Net_Flow_m3h'] = (df_res_h['Net_Power_Deficit_kW'] * 3600) / (rho * cp * delta_t_H)

has_deficit_h = df_res_h['Net_Power_Deficit_kW'] > 1e-6
peak_net_flow_h = df_res_h.loc[has_deficit_h, 'Net_Flow_m3h'].max() if has_deficit_h.any() else 0.0
Vh_volume_m3_h = (peak_net_flow_h * TAU_STABILITY) / (1 - DEAD_ZONE)

# Identify worst index for Table 2 highlight (Heating)
worst_idx_Vh_h = df_res_h.loc[has_deficit_h, 'Net_Flow_m3h'].idxmax() if has_deficit_h.any() else 0

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- HYDRAULIC STABILITY CALCULATION (Cooling Side - Vh_c) ---
df_res_c['Net_Power_Deficit_kW'] = (df_res_c['Total_Demand_C_kW'] - df_res_c['HP_Out_kW']).clip(lower=0)
df_res_c['Net_Flow_m3h'] = (df_res_c['Net_Power_Deficit_kW'] * 3600) / (rho * cp * delta_t_C)

has_deficit_c = df_res_c['Net_Power_Deficit_kW'] > 1e-6
peak_net_flow_c = df_res_c.loc[has_deficit_c, 'Net_Flow_m3h'].max() if has_deficit_c.any() else 0.0
Vh_volume_m3_c = (peak_net_flow_c * TAU_STABILITY) / (1 - DEAD_ZONE)

# Identify worst index for Table 2 highlight (Cooling)
worst_idx_Vh_c = df_res_c.loc[has_deficit_c, 'Net_Flow_m3h'].idxmax() if has_deficit_c.any() else 0

end_total = time.perf_counter()
total_ms = (end_total - start_total) * 1000

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# =================================================================
# 5. STYLED DISPLAY
# =================================================================

# --- STYLED DISPLAY FUNCTION ---
# --- STYLED DISPLAY FUNCTION ---
def apply_styles(row):
    """Highlight rows where the active demand exceeds HP supply."""
    # Since both columns exist in the dataframe, we check which demand is active (non-zero)
    h_dem = row.get('Total_Demand_H_kW', 0)
    c_dem = row.get('Total_Demand_C_kW', 0)
    
    # Pick the active demand for the current row (works for both H and C tables)
    active_demand = max(h_dem, c_dem)
    
    # Logic: If the active demand is greater than HP capacity, highlight the row blue
    if active_demand > row['Max_Supply_kW'] + 1e-4:
        return ['background-color: #002b5e; color: white'] * len(row)
    return [''] * len(row)

def clean_format(x):
    if isinstance(x, (int, float)):
        if abs(x) < 1e-6: return "0"
        if x % 1 == 0: return f"{int(x)}"
        return f"{x:.2f}"
    return str(x)

# =================================================================
# HEATING SIDE SUMMARY
# =================================================================
delta_u_h = cp * delta_t_H 
v_c_h = (TANK_KWH_H_FINAL * 3600 * S) / (rho * delta_u_h * TOTAL_EFFICIENCY)

print(f"HEATING SIDE:")
print(f"Optimized Tank (Vc): {TANK_KWH_H_FINAL:.2f} kWh")
print(f"Charging: {REHEAT_STRATEGY_H.title()} | Discharge: {DISCHARGE_STRATEGY_H.title()}")
print(f"Final Volume (Vc): {v_c_h:.2f} m³")
print(f"Final Volume (Vh): {Vh_volume_m3_h:.2f} m³")

print("-" * 33)
print("ZONE DISCHARGE STRATEGIES (H):")
for i in range(num_zones):
    status = f"MAINTAIN T_SET at {T_SET_LIST[i]}°C" if ZONE_TES_FIRST_H[i] else f"ALLOW DRIFT from {T_SET_LIST[i]}°C down to {final_t_mins[i]:.2f}°C"
    print(f"Zone {i+1}: {status}")
print("-" * 33)

print("\n" + "="*40 + "\n")

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- HEATING SECTION DISPLAY ---
print("\n" + "="*30 + " TABLE 1: ENERGY CAPACITY (Vc_H) " + "="*30)
# Locate the moment of lowest energy in the hot tank
min_tank_h = df_res_h['Tank_kWh'].min()
worst_idx_vc_h = df_res_h[df_res_h['Tank_kWh'] <= min_tank_h + 1e-6].index[0]

# Define window: From last time tank was full before the dip, to next time it's full after
is_full_h = (df_res_h['Tank_kWh'] >= TANK_KWH_H_FINAL - 1e-6)
start_cand_vc_h = df_res_h.index[(df_res_h.index < worst_idx_vc_h) & is_full_h]
start_idx_vc_h = start_cand_vc_h[-1] if not start_cand_vc_h.empty else 0
end_cand_vc_h = df_res_h.index[(df_res_h.index > worst_idx_vc_h) & is_full_h]
end_idx_vc_h = end_cand_vc_h[0] if not end_cand_vc_h.empty else len(df_res_h) - 1

df_zoom_vc_h = df_res_h.iloc[start_idx_vc_h : end_idx_vc_h + 1].copy()
numeric_cols_h = df_zoom_vc_h.select_dtypes(include=['number']).columns
display(df_zoom_vc_h.style.format({col: clean_format for col in numeric_cols_h}).hide().apply(apply_styles, axis=1))

print("\n" + "="*30 + " TABLE 2: HYDRAULIC STABILITY (Vh_H) " + "="*30)
# Center the view on the peak flow event calculated in Section 4.3
start_idx_Vh_h = max(0, worst_idx_Vh_h - 2)
end_idx_Vh_h = min(len(df_res_h)-1, worst_idx_Vh_h + 2)
df_zoom_Vh_h = df_res_h.iloc[start_idx_Vh_h : end_idx_Vh_h + 1].copy()
display(df_zoom_Vh_h.style.format({col: clean_format for col in numeric_cols_h}).hide().apply(lambda x: ['background-color: #5e0000; color: white' if x.name == worst_idx_Vh_h else '' for i in x], axis=1).set_caption("Peak Net Flow Heating Highlighted"))

# =================================================================
# COOLING SIDE SUMMARY
# =================================================================
delta_u_c = cp * delta_t_C 
v_c_c = (TANK_KWH_C_FINAL * 3600 * S) / (rho * delta_u_c * TOTAL_EFFICIENCY)

print(f"COOLING SIDE:")
print(f"Optimized Tank (Vc): {TANK_KWH_C_FINAL:.2f} kWh")
print(f"Charging: {REHEAT_STRATEGY_C.title()} | Discharge: {DISCHARGE_STRATEGY_C.title()}")
print(f"Final Volume (Vc): {v_c_c:.2f} m³")
print(f"Final Volume (Vh): {Vh_volume_m3_c:.2f} m³")

print("-" * 33)
print("ZONE DISCHARGE STRATEGIES (C):")
for i in range(num_zones):
    status = f"MAINTAIN T_SET at {T_SET_LIST[i]}°C" if ZONE_TES_FIRST_C[i] else f"ALLOW DRIFT from {T_SET_LIST[i]}°C up to {final_t_maxs[i]:.2f}°C"
    print(f"Zone {i+1}: {status}")
print("-" * 33)

# -------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# --- COOLING SECTION DISPLAY ---
print("\n" + "="*30 + " TABLE 1: ENERGY CAPACITY (Vc_C) " + "="*30)
min_tank_c = df_res_c['Tank_kWh'].min()
worst_idx_vc_c = df_res_c[df_res_c['Tank_kWh'] <= min_tank_c + 1e-6].index[0]

is_full_c = (df_res_c['Tank_kWh'] >= TANK_KWH_C_FINAL - 1e-6)
start_cand_vc_c = df_res_c.index[(df_res_c.index < worst_idx_vc_c) & is_full_c]
start_idx_vc_c = start_cand_vc_c[-1] if not start_cand_vc_c.empty else 0
end_cand_vc_c = df_res_c.index[(df_res_c.index > worst_idx_vc_c) & is_full_c]
end_idx_vc_c = end_cand_vc_c[0] if not end_cand_vc_c.empty else len(df_res_c) - 1

df_zoom_vc_c = df_res_c.iloc[start_idx_vc_c : end_idx_vc_c + 1].copy()
numeric_cols_c = df_zoom_vc_c.select_dtypes(include=['number']).columns
display(df_zoom_vc_c.style.format({col: clean_format for col in numeric_cols_c}).hide().apply(apply_styles, axis=1))

print("\n" + "="*30 + " TABLE 2: HYDRAULIC STABILITY (Vh_C) " + "="*30)
start_idx_Vh_c = max(0, worst_idx_Vh_c - 2)
end_idx_Vh_c = min(len(df_res_c)-1, worst_idx_Vh_c + 2)
df_zoom_Vh_c = df_res_c.iloc[start_idx_Vh_c : end_idx_Vh_c + 1].copy()
display(df_zoom_Vh_c.style.format({col: clean_format for col in numeric_cols_c}).hide().apply(lambda x: ['background-color: #5e0000; color: white' if x.name == worst_idx_Vh_c else '' for i in x], axis=1).set_caption("Peak Net Flow Cooling Highlighted"))

# export tables
import nest_asyncio
nest_asyncio.apply()

import dataframe_image as dfi

import dataframe_image as dfi

dfi.export(df_zoom_vc_c.style.format({col: clean_format for col in numeric_cols_c}).hide().apply(apply_styles, axis=1), 'table_cooling_vc.pdf', max_cols=-1, table_conversion='matplotlib')
dfi.export(df_zoom_Vh_c.style.format({col: clean_format for col in numeric_cols_c}).hide().apply(lambda x: ['background-color: #5e0000; color: white' if x.name == worst_idx_Vh_c else '' for i in x], axis=1), 'table_cooling_vh.pdf', max_cols=-1, table_conversion='matplotlib')
dfi.export(df_zoom_vc_h.style.format({col: clean_format for col in numeric_cols_h}).hide().apply(apply_styles, axis=1), 'table_heating_vc.pdf', max_cols=-1, table_conversion='matplotlib')
dfi.export(df_zoom_Vh_h.style.format({col: clean_format for col in numeric_cols_h}).hide().apply(lambda x: ['background-color: #5e0000; color: white' if x.name == worst_idx_Vh_h else '' for i in x], axis=1), 'table_heating_vh.pdf', max_cols=-1, table_conversion='matplotlib')

print(f"Performance: {total_ms:.2f} ms ({run_count} runs total)")

Optimizing Hot Tank Size (Bisection Method)...
  Iter 1: Testing 2500.00 kWh -> Pass
  Iter 2: Testing 1250.00 kWh -> Pass
  Iter 3: Testing 625.00 kWh -> Pass
  Iter 4: Testing 312.50 kWh -> Pass
  Iter 5: Testing 156.25 kWh -> Pass
  Iter 6: Testing 78.12 kWh -> Pass
  Iter 7: Testing 39.06 kWh -> Pass
  Iter 8: Testing 19.53 kWh -> Pass
  Iter 9: Testing 9.77 kWh -> Violations
  Iter 10: Testing 14.65 kWh -> Violations
  Iter 11: Testing 17.09 kWh -> Violations
  Iter 12: Testing 18.31 kWh -> Pass
  Iter 13: Testing 17.70 kWh -> Violations
  Iter 14: Testing 18.01 kWh -> Violations
  Iter 15: Testing 18.16 kWh -> Pass

Optimizing Cold Tank Size (Bisection Method)...
  Iter 1: Testing 2500.00 kWh -> Pass
  Iter 2: Testing 1250.00 kWh -> Pass
  Iter 3: Testing 625.00 kWh -> Pass
  Iter 4: Testing 312.50 kWh -> Pass
  Iter 5: Testing 156.25 kWh -> Pass
  Iter 6: Testing 78.12 kWh -> Pass
  Iter 7: Testing 39.06 kWh -> Pass
  Iter 8: Testing 19.53 kWh -> Pass
  Iter 9: Testing 9.77 kWh 

Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
331,111,0,112.16,112.16,18.16,0,1,0.00,19.70,21,0,1.70,36.60,22,0,3.15,16.80,21,0,1.45,21.80,21,0,1.88,16.10,21,0,1.39,STRUCT,0,0
332,83.80,0,66.94,66.94,12.24,0,0,0,11.50,20.96,0,0.79,29.40,22,0,2.53,5.20,20.99,0,0.36,21.70,20.95,0,1.49,16,20.95,0,1.10,Discharge,16.86,1.45
333,82.50,0,67.41,67.41,6.97,0,0,0,11.30,20.92,0,0.80,28.80,22,0,2.48,5,20.98,0,0.35,21.50,20.91,0,1.51,15.90,20.90,0,1.12,Discharge,15.09,1.30
334,82.20,0,72.58,72.58,3.62,0,0,0,11.30,20.90,0,0.86,28.60,22,0,2.46,5,20.97,0,0.38,21.50,20.88,0,1.63,15.80,20.87,0,1.20,Discharge,9.62,0.83
335,82.50,0,72.45,72.45,0.12,0,0,0,11.30,20.88,0,0.85,28.80,22,0,2.48,5,20.96,0,0.38,21.50,20.85,0,1.63,15.90,20.83,0,1.20,Discharge,10.05,0.87
336,114.90,0,115,115,0.12,0,620,180.42,19.20,20.88,402.98,1.66,34.70,22,0,2.99,24,20.96,402.98,2.07,21.30,20.85,402.98,1.84,15.70,20.84,402.98,1.35,STRUCT,0,0
337,103.60,0,115,115,0.12,0,619,1.58,24.70,20.92,3.52,2.33,18,22,0,1.55,24,20.98,3.52,2.16,21.20,20.90,3.52,2.22,15.70,20.89,3.52,1.64,STRUCT,0,0
338,103,0,115,115,0.12,0,618,1.50,24.60,20.96,2.21,2.34,17.80,22,0,1.53,23.90,20.99,2.21,2.16,21.10,20.96,2.21,2.23,15.60,20.95,2.21,1.65,STRUCT,0,0
339,103.60,0,115,115,1.43,0,617,1.58,24.70,21,1.20,2.31,18,22,0,1.55,24,21,1.20,2.15,21.20,21,1.20,2.17,15.70,21,1.20,1.61,STRUCT,0,0
340,104.60,0,115,115,11.83,0,616,1.61,24.90,21,0,2.14,18.40,22,0,1.58,24.10,21,0,2.08,21.40,21,0,1.84,15.80,21,0,1.36,STRUCT,0,0



============================== TABLE 2: HYDRAULIC STABILITY (Vh_H) ==============================


Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
330,109.60,0,112.74,112.74,18.16,0,2,0.00,19.50,21,0,1.68,36,22,0,3.10,16.60,21,0,1.43,21.60,21,0,1.86,15.90,21,0,1.37,STRUCT,0,0
331,111,0,112.16,112.16,18.16,0,1,0.00,19.70,21,0,1.70,36.60,22,0,3.15,16.80,21,0,1.45,21.80,21,0,1.88,16.10,21,0,1.39,STRUCT,0,0
332,83.80,0,66.94,66.94,12.24,0,0,0,11.50,20.96,0,0.79,29.40,22,0,2.53,5.20,20.99,0,0.36,21.70,20.95,0,1.49,16,20.95,0,1.10,Discharge,16.86,1.45
333,82.50,0,67.41,67.41,6.97,0,0,0,11.30,20.92,0,0.80,28.80,22,0,2.48,5,20.98,0,0.35,21.50,20.91,0,1.51,15.90,20.90,0,1.12,Discharge,15.09,1.30
334,82.20,0,72.58,72.58,3.62,0,0,0,11.30,20.90,0,0.86,28.60,22,0,2.46,5,20.97,0,0.38,21.50,20.88,0,1.63,15.80,20.87,0,1.20,Discharge,9.62,0.83


COOLING SIDE:
Optimized Tank (Vc): 0.15 kWh
Charging: Proportional | Discharge: Proportional
Final Volume (Vc): 0.03 m³
Final Volume (Vh): 1.50 m³
---------------------------------
ZONE DISCHARGE STRATEGIES (C):
Zone 1: ALLOW DRIFT from 21.0°C up to 23.00°C
Zone 2: ALLOW DRIFT from 22.0°C up to 22.00°C
Zone 3: ALLOW DRIFT from 21.0°C up to 23.00°C
Zone 4: ALLOW DRIFT from 21.0°C up to 23.00°C
Zone 5: ALLOW DRIFT from 21.0°C up to 23.00°C
---------------------------------

============================== TABLE 1: ENERGY CAPACITY (Vc_C) ==============================


Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
4092,0,63.30,65,65,0.15,0,1,0.00,12.40,21,0,1.33,0,22,0,0,9.90,21,0,1.07,24.50,21,0,2.64,16.50,21,0,1.78,STRUCT,0,0
4093,0,65.70,65,65,0.15,0,0,0,13.10,21.00,0,1.40,0,22,0,0,22.60,21.00,0,2.41,18,21.00,0,1.92,12,21.00,0,1.28,Discharge,0.70,0.08
4094,0,79.90,65,65,0.15,0,0,0,14,21.05,0,1.23,0,22,0,0,23.30,21.05,0,2.04,25.60,21.05,0,2.24,17,21.05,0,1.49,Discharge,14.90,1.60
4095,0,82.30,65,65,0.15,0,0,0,14.40,21.10,0,1.22,0,22,0,0,24,21.11,0,2.04,26.70,21.12,0,2.27,17.20,21.12,0,1.46,Discharge,17.30,1.86
4096,0,101.30,65,65,0.15,0,0,0,31.20,21.30,0,2.16,0,22,0,0,24.80,21.22,0,1.71,27.80,21.23,0,1.92,17.50,21.22,0,1.21,Discharge,36.30,3.91
4097,0,102,65,65,0.15,0,0,0,31.90,21.50,0,2.19,0,22,0,0,25.40,21.32,0,1.74,27.50,21.33,0,1.89,17.20,21.32,0,1.18,Discharge,37,3.98
4098,0,74.30,65,65,0.15,0,0,0,31.90,21.57,0,3.00,0,22,0,0,25.20,21.36,0,2.37,10.90,21.35,0,1.03,6.30,21.34,0,0.59,Discharge,9.30,1.00
4099,0,76.40,65,65,0.15,0,0,0,32.70,21.66,0,3.00,0,22,0,0,25.40,21.40,0,2.33,12.20,21.37,0,1.12,6.10,21.35,0,0.56,Discharge,11.40,1.23
4100,0,108.70,65,65,0.15,0,0,0,56.50,22.06,0,3.64,0,22,0,0,43.50,21.61,0,2.80,5.80,21.39,0,0.37,2.90,21.37,0,0.19,Discharge,43.70,4.70
4101,0,100.60,65,65,0.15,0,0,0,54,22.39,0,3.76,0,22,0,0,41.40,21.78,0,2.88,3.80,21.41,0,0.26,1.40,21.38,0,0.10,Discharge,35.60,3.83



============================== TABLE 2: HYDRAULIC STABILITY (Vh_C) ==============================


Sim_Timestep,Total_Demand_H_kW,Total_Demand_C_kW,Max_Supply_kW,HP_Out_kW,Tank_kWh,Deficit_kW,Window_h,Tsat_TES,Demand_Z1_kW,Temp_Z1,Tsat_Z1_h,Flow_Z1_m3h,Demand_Z2_kW,Temp_Z2,Tsat_Z2_h,Flow_Z2_m3h,Demand_Z3_kW,Temp_Z3,Tsat_Z3_h,Flow_Z3_m3h,Demand_Z4_kW,Temp_Z4,Tsat_Z4_h,Flow_Z4_m3h,Demand_Z5_kW,Temp_Z5,Tsat_Z5_h,Flow_Z5_m3h,Strategy,Net_Power_Deficit_kW,Net_Flow_m3h
4194,0,73.80,65,65,0.15,0,0,0,31.40,21.22,0,2.98,0,22,0,0,24.70,21.12,0,2.34,11.50,21.05,0,1.09,6.20,21.05,0,0.59,Discharge,8.80,0.95
4195,0,68.60,65,65,0.15,0,0,0,30.60,21.25,0,3.12,0,22,0,0,23.80,21.13,0,2.43,9.40,21.06,0,0.96,4.80,21.05,0,0.49,Discharge,3.60,0.39
4196,0,109.70,65,65,0.15,0,0,0,56.10,21.65,0,3.58,0,22,0,0,43.60,21.34,0,2.78,6.50,21.09,0,0.41,3.50,21.08,0,0.22,Discharge,44.70,4.81
4197,0,100.20,65,65,0.15,0,0,0,53.30,21.98,0,3.72,0,22,0,0,41.20,21.51,0,2.88,4,21.10,0,0.28,1.70,21.09,0,0.12,Discharge,35.20,3.79
4198,0,95.30,65,65,0.15,0,0,0,51.50,22.27,0,3.78,0,22,0,0,39.80,21.65,0,2.92,3,21.11,0,0.22,1,21.09,0,0.07,Discharge,30.30,3.26


Performance: 7836.16 ms (1 runs total)
